---
title: Pemetaan sebaran banjir dengan Sentinel-1 di Aceh Utara dan Aceh Timur
short_title: Banjir Aceh Utara
subject: Studi Kasus
subtitle: Memetakan sebaran banjir Siklon Tropis Senyar serta estimasi dampaknya terhadap tutupan lahan dan penduduk.
description: Menggunakan citra radar Sentinel-1 RTC bersama JRC Global Surface Water, Copernicus DEM GLO-30, ESA WorldCover 2021, dan WorldPop 2025 melalui STAC untuk memetakan sebaran banjir November 2025 di pesisir utara Aceh.
authors:
  - name: Muhammad Taufik
    github: taufik-shf
    roles:
      - writing
      - methodology
      - formal analysis
      - software
      - visualization
      - data curation
  - name: Gatot Pramono
    roles:
      - conceptualization
keywords:
  - sentinel-1
  - sar
  - banjir
  - aceh-utara
  - siklon-senyar
  - stac
  - jrc-gsw
  - esa-worldcover
  - worldpop
---

## A. Latar belakang

Hujan lebat yang dipicu Siklon Tropis Senyar pada akhir November 2025 menyebabkan banjir yang meluas di pesisir Kabupaten Aceh Utara hingga perbatasan Aceh Timur. Kejadian banjir umumnya diiringi tutupan awan tebal yang menghalangi pengamatan satelit optis seperti Sentinel-2 dan Landsat. Citra *Synthetic Aperture Radar* (SAR) dari Sentinel-1 sering dipakai untuk pemetaan cepat genangan banjir karena gelombang radarnya mampu menembus awan dan merekam permukaan bumi pada siang maupun malam hari [(UN-SPIDER, 2023)](#ref-un-spider).

:::{figure} ../../../assets/sar_backscatter_banjir.svg
:name: fig-sar-backscatter
:width: 100%

Ilustrasi mekanisme hamburan balik (*backscatter*) radar SAR pita-C pada berbagai permukaan daratan dan genangan.
:::

Deteksi genangan banjir pada citra SAR didasarkan pada perbedaan hamburan balik (*backscatter*) antarjenis permukaan. Daratan yang kasar menyebarkan sebagian energi radar kembali ke sensor, sedangkan permukaan air yang tenang memantulkan sinyal menjauh dari sensor secara spekular (*specular reflection*) sehingga tampak gelap dengan nilai hamburan balik rendah [(Twele dkk., 2016)](#ref-twele-2016). Dengan membandingkan citra saat banjir terhadap kondisi dasar (*baseline*) sebelum banjir pada geometri orbit yang sama, area yang baru tergenang dapat dibedakan dari tutupan lahan normal.

Untuk memetakan banjir dan memperkirakan dampaknya, analisis ini memadukan citra Sentinel-1 *Radiometrically Terrain Corrected* (RTC) dengan data perairan permanen JRC *Global Surface Water*, model elevasi Copernicus DEM GLO-30, peta tutupan lahan ESA WorldCover 2021, serta data kependudukan WorldPop 2025. Sebagian besar dataset diakses melalui katalog STAC (*SpatioTemporal Asset Catalog*) eksternal menggunakan `pystac_client` dan `odc.stac`.

## B. Menyiapkan data dan wilayah kajian

In [ ]:
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr
import pandas as pd
import rioxarray as rxr
from ipyleaflet import basemaps
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch
from odc.stac import load
from planetary_computer import sign_url
from pystac_client import Client
from rasterio.enums import Resampling
from scipy import ndimage
from shapely.geometry import box
from skimage.filters import threshold_otsu

Seluruh dataset diakses langsung sebagai *Cloud-Optimized GeoTIFF* (COG) dari katalog STAC Microsoft Planetary Computer sesuai batas wilayah kajian.

In [ ]:
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
client = Client.open(stac_url)

Wilayah kajian (*Area of Interest*, AoI) mencakup pesisir dan dataran rendah Aceh Utara hingga sebagian Aceh Timur.

In [ ]:
min_lon, max_lon = 97.118333, 97.763779
min_lat, max_lat = 4.828044, 5.377917
bbox = (min_lon, min_lat, max_lon, max_lat)

aoi = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")
aoi.explore(tiles=basemaps.Esri.WorldImagery)

## C. Memilih orbit dan menyusun kondisi dasar

Analisis perubahan dilakukan dengan membandingkan kondisi dasar (*baseline*) sebelum banjir dengan kondisi saat kejadian banjir pada akhir November 2025. Ketersediaan rekaman `sentinel-1-rtc` dari akhir Oktober hingga awal Desember 2025 diperiksa terlebih dahulu karena satu wilayah dapat tercakup oleh beberapa jalur satelit dengan arah lintasan (*ascending* atau *descending*) dan nomor orbit relatif (*relative orbit*) yang berbeda.

Rekaman yang ditemukan kemudian dikelompokkan berdasarkan arah lintasan (`orbit_state`) dan nomor orbit relatif untuk melihat jumlah citra serta rentang tanggal tiap jalur. Pengelompokan ini diperlukan agar citra pra-banjir dan saat banjir dipilih dari jalur dengan arah gerak serta sudut pandang radar yang sama, sehingga perbedaan geometri pengamatan tidak terbaca sebagai perubahan permukaan.

In [ ]:
# Cari seluruh item sentinel-1-rtc pada bbox dan rentang waktu yang ditentukan
all_s1 = client.search(
    collections=["sentinel-1-rtc"],
    bbox=bbox,
    datetime="2025-10-20/2025-12-05",
).item_collection()

# Ekstrak atribut orbit dan tanggal akuisisi dari properti STAC tiap item
records = [
    {
        "date": item.datetime.strftime("%Y-%m-%d"),
        "orbit_state": item.properties.get("sat:orbit_state"),
        "relative_orbit": item.properties.get("sat:relative_orbit"),
        "platform": item.properties.get("platform"),
    }
    for item in all_s1
]

# Rekap jumlah item serta tanggal awal-akhir per arah lintasan dan nomor orbit relatif
orbit_summary = (
    pd.DataFrame(records)
    .groupby(["orbit_state", "relative_orbit"])
    .agg(jumlah_item=("date", "count"), tanggal_awal=("date", "min"), tanggal_akhir=("date", "max"))
    .reset_index()
)
orbit_summary

Dari tabel ringkasan di atas, kombinasi `ascending` dengan `relative_orbit = 143` memiliki jumlah rekaman terbanyak (`12` item) dalam rentang `2025-10-23` hingga `2025-12-04`. Orbit ini dipilih untuk menyusun kondisi dasar pra-banjir (`2025-10-23` sampai `2025-11-22`) dan mengambil citra saat banjir pada `2025-11-28`.

In [ ]:
relative_orbit = 143
pre_flood_range = "2025-10-23/2025-11-22"  # Rentang baseline sebelum banjir
flood_date = "2025-11-26/2025-11-30"       # Tanggal kejadian siklon

# Cari item sentinel-1-rtc pada rentang pra-banjir dan batasi hanya ke orbit relatif 143
items_pre = client.search(
    collections=["sentinel-1-rtc"],
    bbox=bbox,
    datetime=pre_flood_range,
    query={"sat:relative_orbit": {"eq": relative_orbit}},
).item_collection()

# Cari item pada tanggal banjir dengan orbit yang sama agar geometri konsisten
items_flood = client.search(
    collections=["sentinel-1-rtc"],
    bbox=bbox,
    datetime=flood_date,
    query={"sat:relative_orbit": {"eq": relative_orbit}},
).item_collection()

print(f"Item pra-banjir: {len(items_pre)}")
print(f"Item saat banjir: {len(items_flood)}")

Polarisasi VV dimuat pada proyeksi UTM zona 47N (`EPSG:32647`) dengan resolusi 20 meter. Polarisasi VV sensitif terhadap perubahan kekasaran permukaan air sehingga umum dipakai untuk mendeteksi genangan terbuka [(Twele dkk., 2016)](#ref-twele-2016).

In [ ]:
crs = "EPSG:32647"
resolution = 20

kwargs = dict(
    bands=["vv"],
    crs=crs,
    resolution=resolution,
    bbox=bbox,
    groupby="solar_day",
    patch_url=sign_url,
    chunks={"time": 1, "x": 2048, "y": 2048},
)

# Memuat data dengan keyword arguments di atas
ds_pre = load(items_pre, **kwargs)["vv"] # baseline
ds_flood = load(items_flood, **kwargs)["vv"] # kejadian

print("Pra-banjir:", [str(t)[:10] for t in ds_pre.time.values])
print("Saat banjir:", [str(t)[:10] for t in ds_flood.time.values])

Tanggal-tanggal yang termuat sudah sesuai untuk analisis perbandingan, sehingga data dapat mulai diproses. Sebelum selisih perubahan *backscatter* ($\Delta\text{VV}$) dihitung, tahap *preprocessing* citra SAR dilakukan terlebih dahulu untuk membersihkan piksel tidak valid, menyusun komposit median pra-banjir, mengonversi nilai ke skala desibel ($\text{dB}$), dan meredam *speckle*.

In [ ]:
# Buang nilai tidak valid sebelum komposit median agar piksel kosong tidak memengaruhi hasil
pre_median = ds_pre.where((ds_pre > 0) & (ds_pre != -32768)).median(dim="time").compute()
flood_linear = ds_flood.where((ds_flood > 0) & (ds_flood != -32768)).median(dim="time").compute()

# Konversi ke dB karena analisis perubahan dan ambang genangan menggunakan satuan desibel
pre_db = 10 * np.log10(pre_median)
flood_db = 10 * np.log10(flood_linear)


def apply_median_filter(da, size=3):
    # Filter median 3x3 piksel (60x60 m) untuk meredam derau speckle pada citra SAR
    filtered = ndimage.median_filter(da.values.astype(np.float32), size=size)
    return da.copy(data=filtered)


vv_baseline = apply_median_filter(pre_db, size=3)
vv_flood = apply_median_filter(flood_db, size=3)

# Selisih negatif menunjukkan penurunan hamburan balik dibanding kondisi dasar
vv_change = vv_flood - vv_baseline

## D. Mendeteksi genangan dan menyaring gangguan

Suatu piksel ditandai sebagai calon genangan jika nilai *backscatter* saat banjir tergolong rendah dan turun lebih dari `3 dB` dibanding sebelum banjir (`vv_change < -3`). Nilai batas *backscatter* untuk memisahkan permukaan air dan daratan dihitung secara otomatis dari `vv_flood` menggunakan metode Otsu [(Otsu, 1979)](#ref-otsu-1979); [(Markert dkk., 2020)](#ref-markert-2020), sedangkan syarat penurunan `3 dB` mencegah area yang sejak awal sudah bernilai rendah ikut terhitung sebagai banjir.

In [ ]:
# Hitung batas backscatter permukaan air secara otomatis dari citra saat banjir
valid_vv = vv_flood.values[np.isfinite(vv_flood.values)]
vv_threshold = float(threshold_otsu(valid_vv))

# Syarat penurunan > 3 dB dibanding kondisi dasar
change_threshold = -3.0

flood_raw = (vv_flood < vv_threshold) & (vv_change < change_threshold)
print(f"Ambang Otsu (VV): {vv_threshold:.2f} dB")

Badan air lain (seperti laut, sungai besar, dan tambak pesisir) masih ikut terdeteksi oleh kriteria tersebut. Agar fokus pada genangan banjir, perairan permanen (`seasonality >= 10` bulan dalam setahun) perlu disaring keluar menggunakan data *JRC Global Surface Water* (`jrc-gsw`)[(Pekel dkk., 2016)](#ref-pekel-2016).

In [ ]:
# Menggunakan client STAC yang sama untuk mencari koleksi jrc-gsw
items_gsw = client.search(collections=["jrc-gsw"], bbox=bbox).item_collection()

# Memuat gsw
seasonality = load(
    items_gsw,
    bands=["seasonality"],
    geobox=vv_flood.odc.geobox,
    resampling="nearest",
    patch_url=sign_url,
)["seasonality"].isel(time=0).compute()

permanent_water = seasonality >= 10

# Pertahankan piksel yang bukan perairan permanen
flood_no_water = flood_raw & ~permanent_water

In [ ]:
# Fungsi bantu untuk plotting
def plot_map(ax, base, title="", overlays=None, legend=None, cmap="gray", vmin=-22, vmax=0, cbar_label=None):
    base.plot.imshow(
        ax=ax,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=bool(cbar_label),
        add_labels=False,
        cbar_kwargs={"label": cbar_label} if cbar_label else None,
    )
    for mask, mask_cmap, alpha in overlays or []:
        cmap_obj = ListedColormap([mask_cmap]) if isinstance(mask_cmap, str) and mask_cmap.startswith("#") else mask_cmap
        mask.where(mask).plot.imshow(
            ax=ax, cmap=cmap_obj, vmin=0, vmax=1, alpha=alpha, add_colorbar=False, add_labels=False
        )
    ax.set_title(title)
    ax.tick_params(axis="both", labelsize=8)
    ax.locator_params(axis="x", nbins=4)
    if legend:
        ax.legend(handles=[Patch(facecolor=c, label=t) for c, t in legend], loc="lower left", fontsize=9)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_map(axes[0], vv_flood, "Sebelum JRC GSW", [(flood_raw, "#38bdf8", 0.75)])
plot_map(axes[1], vv_flood, "Setelah Masking JRC GSW", [(flood_no_water, "#38bdf8", 0.8)])

plt.tight_layout()
plt.show()

Kemudian pada lahan berlereng curam, bayangan radar dapat menghasilkan nilai *backscatter* rendah yang menyerupai genangan air. Untuk menghindarinya, kemiringan lereng dihitung dari **Copernicus DEM GLO-30** (`cop-dem-glo-30`, resolusi 30 meter) lalu diselaraskan ke grid SAR agar hanya area landai (`slope < 5°`) yang dipertahankan [(ESA, 2022)](#ref-cop-dem).

In [ ]:
items_dem = client.search(collections=["cop-dem-glo-30"], bbox=bbox).item_collection()

dem = load(
    items_dem,
    bands=["data"],
    crs=crs,
    resolution=30,
    bbox=bbox,
    patch_url=sign_url,
)["data"].isel(time=0).compute()

# Hitung gradien elevasi per jarak piksel 30 m untuk memperoleh kemiringan dalam derajat
dy, dx = np.gradient(dem.values.astype(np.float32), 30.0, 30.0)
slope_deg = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# Samakan grid mask biner (< 5°) ke geobox SAR dengan nearest agar tetap 0/1
gentle_slope = dem.copy(data=(slope_deg < 5.0).astype(np.uint8)).odc.reproject(
    vv_flood.odc.geobox, resampling="nearest"
) == 1

flood_final = flood_no_water & gentle_slope

In [ ]:
pixel_area_km2 = (resolution * resolution) / 1_000_000
print(f"Sebelum masking lereng : {float(flood_no_water.sum()) * pixel_area_km2:.2f} km²")
print(f"Setelah masking lereng : {float(flood_final.sum()) * pixel_area_km2:.2f} km²")

## E. Analisis hasil dan estimasi dampak

### E.1 Sebaran genangan

Peta berikut menunjukkan sebaran akhir genangan banjir pada 28 November 2025.

In [ ]:
total_flood_km2 = float(flood_final.sum()) * pixel_area_km2
flood_dates = ", ".join(str(t)[:10] for t in ds_flood.time.values)

fig, ax = plt.subplots(figsize=(9, 7))
plot_map(
    ax,
    vv_flood,
    f"Sebaran genangan ({flood_dates}), Luas = {total_flood_km2:.1f} km²",
    [(flood_final, "#38bdf8", 0.8)],
)

plt.tight_layout()
plt.show()

Genangan banyak terkonsentrasi di dataran utara dari barat hingga timur serta di sepanjang aliran sungai bagian tengah. Nilai ambang Otsu (`-12,69 dB`) mengikuti sebaran data di wilayah kajian dan batas `-3 dB` belum dikalibrasi dengan data lapangan, sehingga angka luas di atas bersifat indikatif.

### E.2 Estimasi dampak tutupan lahan (ESA WorldCover)

Untuk melihat tutupan lahan yang terdampak, peta **ESA WorldCover 2021** (`esa-worldcover`, resolusi 10 meter) dimuat pada grid yang sama dengan peta genangan [(Zanaga dkk., 2022)](#ref-zanaga-2022).

In [ ]:
items_wc = client.search(
    collections=["esa-worldcover"],
    bbox=bbox,
    datetime="2021",
).item_collection()

worldcover = load(
    items_wc,
    bands=["map"],
    geobox=vv_flood.odc.geobox,
    resampling="nearest",  # Nearest menjaga kode kelas tutupan lahan tidak tercampur
    patch_url=sign_url,
)["map"].isel(time=0).compute()

# Baca daftar kode dan nama kelas langsung dari metadata STAC
wc_classes = {
    c["value"]: c["description"]
    for c in items_wc[0].assets["map"].extra_fields["classification:classes"]
}
wc_classes

Luas total dan luas yang tergenang pada setiap kelas dihitung langsung menggunakan `value_counts()`.

In [ ]:
pixel_area_ha = (resolution * resolution) / 10_000

total_ha = pd.Series(worldcover.values.ravel()).value_counts() * pixel_area_ha
flooded_ha = pd.Series(worldcover.values[flood_final.values]).value_counts() * pixel_area_ha

impact_table = pd.DataFrame(
    {
        "Tutupan lahan": total_ha.index.map(wc_classes),
        "Terdampak (ha)": flooded_ha.reindex(total_ha.index, fill_value=0).round(1),
        "Total AoI (ha)": total_ha.round(1),
    }
)
impact_table["Proporsi (%)"] = (
    impact_table["Terdampak (ha)"] / impact_table["Total AoI (ha)"] * 100
).round(1)
impact_table.sort_values("Terdampak (ha)", ascending=False)

Dari tabel di atas, lahan pertanian (*Cropland*) menjadi tutupan lahan yang paling luas terdampak banjir, mencapai lebih dari separuh total lahan pertanian di wilayah kajian. Sebaran lahan pertanian yang tergenang dipetakan di bawah ini.

In [ ]:
cropland_affected = (worldcover == 40) & flood_final
crop_ha = float(cropland_affected.sum()) * pixel_area_ha

fig, ax = plt.subplots(figsize=(9, 7))
plot_map(
    ax,
    vv_flood,
    f"Lahan pertanian terdampak ({crop_ha:,.0f} ha)",
    [(flood_final, "#38bdf8", 0.35), (cropland_affected, "#84cc16", 0.85)],
    legend=[("#38bdf8", "Sebaran genangan"), ("#84cc16", "Lahan pertanian tergenang")],
)

plt.tight_layout()
plt.show()

Pada peta terlihat bagian barat hingga tengah genangan didominasi lahan pertanian, sedangkan jalur pesisir utara yang tetap berwarna biru merupakan perairan dan lahan basah. Perlu dicatat bahwa peta WorldCover berasal dari tahun 2021 sehingga perubahan penggunaan lahan setelahnya belum tercermin.

### E.3 Estimasi penduduk terdampak (WorldPop)

Selain tutupan lahan, sebaran genangan dapat digabungkan dengan data kependudukan **WorldPop 2025** (resolusi 100 meter) untuk memperkirakan jumlah warga yang tinggal di area terdampak [(WorldPop, 2025)](#ref-worldpop-2025). Data dibaca langsung dari berkas COG publik Piksel sesuai batas AoI.

In [ ]:
worldpop_url = "https://piksel-production-public-data.s3.ap-southeast-3.amazonaws.com/worldpop_idn_100m/1.0.0/2025--P1Y/idn_pop_2025_CN_100m_R2025A_v1.tif"

pop = (
    rxr.open_rasterio(worldpop_url, masked=True)
    .squeeze("band", drop=True)
    .rio.clip_box(*bbox)
    .compute()
)
pop = pop.where(pop > 0)

Data WorldPop menyimpan jumlah penduduk per sel 100 meter (`EPSG:4326`), sehingga tidak bisa dipecah ke resolusi 20 meter tanpa mengubah total jiwanya. Oleh karena itu, peta banjir yang disesuaikan ke grid WorldPop.

In [ ]:
# Kosongkan nodata -32768 warisan Sentinel-1 sebelum reproject agar tidak bentrok tipe data
flood_100m = (
    flood_final.astype("float32")
    .rio.write_nodata(np.nan)
    .rio.reproject_match(pop, resampling=Resampling.nearest)
    == 1
)

pop_exposed = pop.where(flood_100m)

total_pop = float(pop.sum())
exposed_pop = float(pop_exposed.sum())
print(f"Total penduduk di AoI  : {total_pop:,.0f} jiwa")
print(f"Perkiraan terdampak    : {exposed_pop:,.0f} jiwa ({exposed_pop / total_pop * 100:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

flood_100m.where(flood_100m).plot.imshow(
    ax=ax, cmap=ListedColormap(["#38bdf8"]), vmin=0, vmax=1, alpha=0.35, add_colorbar=False, add_labels=False
)
pop_exposed.plot.imshow(
    ax=ax,
    cmap="YlOrRd",
    norm=LogNorm(vmin=1, vmax=float(pop_exposed.max())),
    add_labels=False,
    cbar_kwargs={"label": "Jiwa per sel 100 m"},
)

ax.set_title(f"Perkiraan penduduk terdampak ({exposed_pop:,.0f} jiwa)")
ax.tick_params(axis="both", labelsize=8)
ax.locator_params(axis="x", nbins=4)
plt.tight_layout()
plt.show()

Sekitar 32 ribu jiwa (4,4% dari total penduduk di AoI) berada di sel yang terdeteksi banjir. Angka ini kemungkinan lebih rendah dari kondisi sebenarnya karena genangan di kawasan bangunan rapat sulit terdeteksi dari nilai *backscatter* rendah (lihat [](#fig-sar-backscatter)), serta pencocokan pusat sel 100 meter dapat melewatkan sel permukiman yang hanya tergenang sebagian.

### E.4 Peta interaktif sebaran dan dampak banjir

Seluruh hasil pemetaan (sebaran banjir, lahan pertanian terdampak, dan penduduk terdampak) digabungkan dalam satu peta interaktif.

In [ ]:
m = aoi.explore(
    tiles=basemaps.Esri.WorldImagery,
    name="Batas AoI",
    style_kwds={"fill": False, "color": "white", "weight": 2},
)

flood_final.where(flood_final).astype("float32").odc.explore(
    map=m,
    name="Sebaran genangan",
    cmap=ListedColormap(["#38bdf8"]),
    vmin=0,
    vmax=1,
    opacity=0.75,
)
cropland_affected.where(cropland_affected).astype("float32").odc.explore(
    map=m,
    name="Lahan pertanian terdampak",
    cmap=ListedColormap(["#84cc16"]),
    vmin=0,
    vmax=1,
    opacity=0.85,
)
pop_exposed.odc.explore(
    map=m,
    name="Penduduk terdampak",
    cmap="autumn_r",
    vmin=1,
    vmax=50,
    opacity=0.9,
)
folium.LayerControl(collapsed=False).add_to(m)
m

## F. Pengembangan analisis untuk pemantauan banjir

Penggunaan metode ini dapat diterapkan untuk memantau kejadian banjir di wilayah atau periode lain, maupun diperluas ke beberapa langkah berikut:

- **Memantau lama genangan hingga surut.** Orbit relatif `143` merekam ulang wilayah ini pada 4 dan 10 Desember 2025, sehingga durasi genangan dapat ikut dipetakan.
- **Menguji batas ambang lokal dan validasi lapangan.** Metode Otsu global sering meleset jika area air hanya sedikit, misalnya di wilayah daratan tanpa laut. Pendekatan *edge Otsu* yang dicocokkan dengan data kejadian atau foto udara membantu menjaga ketelitiannya [(Markert dkk., 2020)](#ref-markert-2020).
- **Memverifikasi dengan citra optis setelah awan terbuka.** Saat tutupan awan berkurang pascasiklon, citra Sentinel-2 membantu memeriksa bekas genangan sekaligus membedakan area tambak dari lahan pertanian yang terendam.
- **Menggunakan data tutupan lahan dan kependudukan yang lebih mutakhir dan rinci.** Estimasi dampak akan jauh lebih tepat jika memakai peta tutupan lahan lokal terbaru serta data kependudukan tingkat desa atau kecamatan (termasuk fasilitas umum), daripada hanya mengandalkan dataset global.
- **Mendeteksi genangan di kawasan permukiman.** Penggunaan data koherensi InSAR (produk SLC Sentinel-1) atau kombinasi polarisasi VV dan VH lebih peka terhadap banjir di antara bangunan dibandingkan intensitas VV saja.
- **Memodelkan kedalaman genangan.** Untuk memperkirakan kedalaman air atau *Height Above Nearest Drainage* (HAND), Copernicus DEM (DSM) perlu diganti dengan *Digital Terrain Model* (DTM) seperti FABDEM agar tinggi pohon dan bangunan tidak terbaca sebagai permukaan tanah.

## Referensi

(ref-un-spider)=
1. UN-SPIDER. (2023). *Recommended Practice: Radar-based Flood Mapping*. United Nations Office for Outer Space Affairs (UNOOSA). https://un-spider.org/advisory-support/recommended-practices/recommended-practice-flood-mapping

(ref-twele-2016)=
2. Twele, A., Cao, W., Plank, S., & Martinis, S. (2016). *Sentinel-1-based flood mapping: a fully automated processing chain*. *International Journal of Remote Sensing*, 37(13), 2990–3004. https://doi.org/10.1080/01431161.2016.1192304

(ref-otsu-1979)=
3. Otsu, N. (1979). *A Threshold Selection Method from Gray-Level Histograms*. *IEEE Transactions on Systems, Man, and Cybernetics*, 9(1), 62–66. https://doi.org/10.1109/TSMC.1979.4310076

(ref-markert-2020)=
4. Markert, K. N., Markert, A. M., Mayer, T., Nauman, C., Haag, A., Poortinga, A., Bhandari, B., Thwal, N. S., Kunlamai, T., Chishtie, F., Kwant, M., Phongsapan, K., Clinton, N., Towashiraporn, P., & Saah, D. (2020). *Comparing Sentinel-1 Surface Water Mapping Algorithms and Radiometric Terrain Correction Processing in Southeast Asia Utilizing Google Earth Engine*. *Remote Sensing*, 12(15), 2469. https://doi.org/10.3390/rs12152469

(ref-pekel-2016)=
5. Pekel, J.-F., Cottam, A., Gorelick, N., & Belward, A. S. (2016). *High-resolution mapping of global surface water and its long-term changes*. *Nature*, 540(7633), 418–422.

(ref-cop-dem)=
6. European Space Agency (ESA). (2022). *Copernicus Global Digital Elevation Model (GLO-30)*. Distributed by OpenTopography / Microsoft Planetary Computer.

(ref-zanaga-2022)=
7. Zanaga, D., Van De Kerchove, R., Daems, D., De Keersmaecker, W., Brockmann, C., Kirches, G., Wevers, J., Cartus, O., Santoro, M., Fritz, S., Lesiv, M., Herold, M., Tsendbazar, N.-E., Xu, P., Ramoino, F., & Arino, O. (2022). *ESA WorldCover 10 m 2021 v200*. Zenodo. https://doi.org/10.5281/zenodo.7254221

(ref-worldpop-2025)=
8. WorldPop. (2025). *Global 100m Constrained Population Estimate R2025A v1 (Indonesia)*. University of Southampton. https://hub.worldpop.org/